# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sumit07-git/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

## 1. Ranked actions + reason codes

The action queue ranks content items using the model's estimated probability
of the observed down trend label.

Each recommendation includes a reason code so that a human reviewer can
understand why an item was prioritized.

The queue is intended as directional decision-support, not an automatic
instruction to change content.

In [18]:
!git clone https://github.com/Sumit07-git/flyrank-ml-internship.git

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [19]:
!find /content/flyrank-ml-internship -name "content_refresh_anonymized.csv"

/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv


In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np

df = pd.read_csv(
    "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"
)

print("Rows:", len(df))


Rows: 30000


In [21]:
df["is_declining"] = (
    df["trend_direction"] == "down"
).astype(int)

features = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "engagement_rate"
]

target = "is_declining"

print("Target distribution:")
print(df[target].value_counts())

Target distribution:
is_declining
1    16262
0    13738
Name: count, dtype: int64


In [22]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)

model.fit(
    df[features],
    df[target]
)

df["model_score"] = model.predict_proba(
    df[features]
)[:, 1]

print("Scoring completed.")

Scoring completed.


In [23]:
def get_reason_code(row):
    reasons = []

    if row["model_score"] >= 0.80:
        reasons.append("HIGH_DECLINE_SCORE")

    if row["days_since_last_update"] >= 180:
        reasons.append("STALE_CONTENT")

    if row["impressions_90d"] >= 500:
        reasons.append("VISIBLE_CONTENT")

    if row["avg_position"] >= 10:
        reasons.append("WEAKER_SEARCH_POSITION")

    if row["engagement_rate"] < df["engagement_rate"].median():
        reasons.append("LOWER_ENGAGEMENT")

    if not reasons:
        reasons.append("MODEL_REVIEW")

    return " | ".join(reasons)

In [24]:
def get_action(row):
    if (
        row["model_score"] >= 0.80
        and row["days_since_last_update"] >= 180
        and row["impressions_90d"] >= 500
    ):
        return "PRIORITIZE_REFRESH"

    if row["model_score"] >= 0.60:
        return "REVIEW"

    return "MONITOR"

In [25]:
queue = df.copy()

queue["reason_code"] = queue.apply(
    get_reason_code,
    axis=1
)

queue["recommended_action"] = queue.apply(
    get_action,
    axis=1
)

queue = queue.sort_values(
    "model_score",
    ascending=False
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

print("Action queue created:", queue.shape)

Action queue created: (30000, 49)


In [26]:
queue_columns = [
    "rank",
    "content_id",
    "model_score",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "reason_code",
    "recommended_action"
]

display(
    queue[queue_columns].head(20)
)

,rank,content_id,model_score,days_since_last_update,impressions_90d,avg_position,reason_code,recommended_action
0,1,content_baa7232f4846,1.000000,20,239,7.8,HIGH_DECLINE_SCORE,REVIEW
1,2,content_1b74b27f23b9,1.000000,104,408,2.9,HIGH_DECLINE_SCORE,REVIEW
2,3,content_3f7335f23833,1.000000,20,1073,5.2,HIGH_DECLINE_SCORE | VISIBLE_CONTENT,REVIEW
3,4,content_d5b833d82e72,1.000000,20,1220,7.5,HIGH_DECLINE_SCORE | VISIBLE_CONTENT,REVIEW
4,5,content_20b682542a31,1.000000,20,339,3.3,HIGH_DECLINE_SCORE,REVIEW
5,6,content_aa9b17a4ea86,1.000000,20,310,2.9,HIGH_DECLINE_SCORE,REVIEW
6,7,content_096c9f804d01,1.000000,19,954,2.7,HIGH_DECLINE_SCORE | VISIBLE_CONTENT,REVIEW
7,8,content_03c957094bba,1.000000,20,9449,3.1,HIGH_DECLINE_SCORE | VISIBLE_CONTENT,REVIEW
8,9,content_e1bdad640bcf,1.000000,20,1532,8.0,HIGH_DECLINE_SCORE | VISIBLE_CONTENT,REVIEW
9,10,content_9412afc8a975,1.000000,104,306,22.5,HIGH_DECLINE_SCORE | WEAKER_SEARCH_POSITION,REVIEW


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## 2. Intended use and limits

The action playbook is intended for content or SEO teams that need to
prioritize which content items deserve human review first.

The model score provides directional prioritization based on the observed
training data. Reason codes make the recommendation easier to inspect.

The queue should not be treated as an automatic content-change system.
It does not prove that refreshing a page will improve search performance,
and it does not establish causality or reproduce Google's ranking system.

The recommendations are specific to the dataset, features, label
definition, and validation design used in this project.

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

action_summary = (
    queue["recommended_action"]
    .value_counts()
    .rename_axis("recommended_action")
    .reset_index(name="count")
)

display(action_summary)


,recommended_action,count
0,REVIEW,15984
1,MONITOR,14002
2,PRIORITIZE_REFRESH,14


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## 3. Human review + the no-go list

Before acting on a recommendation, a human reviewer should check the
content item's actual business purpose, search intent, current quality,
freshness, and whether the observed performance pattern has a reasonable
content explanation.

The model should support prioritization rather than replace editorial
judgment.

### Human review checklist

- Confirm the page still serves a valid search intent.
- Check whether the content is actually outdated.
- Review recent changes that may explain the observed performance.
- Check whether the page is important to the site's goals.
- Review the recommendation together with the reason codes.

### No-go list

The system should never automatically publish, delete, merge, or rewrite
content based only on the model score.

It should also not make claims about Google's ranking algorithm or guarantee
that a recommended refresh will increase traffic, rankings, or clicks.

In [28]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

review_checklist = pd.DataFrame({
    "Human check": [
        "Search intent",
        "Content freshness",
        "Recent performance context",
        "Business importance",
        "Reason-code review"
    ],
    "Required before action": [
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes"
    ]
})

display(review_checklist)

,Human check,Required before action
0,Search intent,Yes
1,Content freshness,Yes
2,Recent performance context,Yes
3,Business importance,Yes
4,Reason-code review,Yes


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## 4. Monitoring / retrain triggers

The recommendations should be reviewed if the underlying data changes
substantially or if measured ranking performance deteriorates.

Potential retraining triggers include:

- Precision@K falls materially below the validation result.
- The distribution of model scores changes substantially.
- Feature distributions change substantially.
- The observed down label rate changes substantially.
- New data fields or measurement definitions are introduced.

These triggers indicate that the current playbook may no longer represent
the observed data well. They are monitoring rules rather than guarantees
of model failure.

In [29]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Model score summary:")
display(
    queue["model_score"].describe()
)


Model score summary:


,model_score
count,30000.000000
mean,0.540816
std,0.349750
min,0.000000
25%,0.196667
50%,0.733333
75%,0.870000
max,1.000000


In [30]:
print(
    "Observed declining-label rate:",
    queue["is_declining"].mean()
)

Observed declining-label rate: 0.5420666666666667


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

## 5. Exports for the paper

The ranked action queue and supporting summary tables are exported to
`work/outputs/` so that the deployed research paper can reuse the same
results.

In [31]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os

os.makedirs(
    "flyrank-ml-internship/work/outputs",
    exist_ok=True
)

output_path = (
    "flyrank-ml-internship/work/outputs/"
    "content_action_queue.csv"
)

queue[queue_columns].to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)


Saved: flyrank-ml-internship/work/outputs/content_action_queue.csv


In [32]:
summary_path = (
    "flyrank-ml-internship/work/outputs/"
    "action_summary.csv"
)

action_summary.to_csv(
    summary_path,
    index=False
)

print("Saved:", summary_path)

Saved: flyrank-ml-internship/work/outputs/action_summary.csv


In [33]:
top50_path = (
    "flyrank-ml-internship/work/outputs/"
    "top50_recommendations.csv"
)

queue[queue_columns].head(50).to_csv(
    top50_path,
    index=False
)

print("Saved:", top50_path)

Saved: flyrank-ml-internship/work/outputs/top50_recommendations.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.